# M3 step 4: using the final model

The first three notebooks train M3. This one shows how other parts of the project can use the finished model, for example the RQ3 experiments, the AI edit tests and the demo.

The code lives in `m3_watermark.py`, which has to sit in the same folder as this notebook. Three functions are all most people need. `load_m3` loads the trained model from `best.pt`. `embed` takes a photo, a verdict and the detector's confidence, and returns a stamped copy of the photo. `extract` reads the stamp back and tells you whether it is valid.

## How full-size photos are handled

The model was trained on 128 by 128 pixel photos, but real photos are much bigger. Shrinking every photo to 128 pixels before stamping would throw away most of its detail, and the detectors in RQ3 would then be looking at a blurry image instead of the real one.

So `embed` works out the stamp on a 128 pixel copy of the photo, enlarges only the stamp to the photo's real size, and adds it on top. The photo keeps all its detail and only gains the small stamp. To read it, `extract` shrinks the photo back to 128 pixels and passes it to the decoder. The final model was trained on resized photos, so the extra shrinking does not hurt.

This notebook checks that this works on the 500 test photos at their original size.

## Setup

`MODEL_PATH` points to the final checkpoint from notebook 3. The test photos are the same 500 as before, used at their original size this time.

In [ ]:
import io
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from m3_watermark import load_m3, embed, extract, read_bits, encode_payload, decode_payload

MODEL_PATH = Path(r'E:\VeriStamp_M3\hidden_runs\hidden_unet_jpegresize_v2_10k\best.pt')
TEST_DIR = Path(r'E:\VeriStamp_M3\hidden_data\test')

m3 = load_m3(MODEL_PATH)
test_files = sorted(TEST_DIR.glob('*.jpg'))
print('Loaded model from epoch', m3.epoch, 'on', m3.device)
print('Test photos:', len(test_files))

## Stamp one photo

Stamp a photo with an example verdict, save it as a PNG file, open the file again and read the stamp back.

In [ ]:
photo = Image.open(test_files[0]).convert('RGB')
stamped = embed(m3, photo, verdict='AI', p_ai=0.91, image_tag=123)

stamped.save('example_stamped.png')
reloaded = Image.open('example_stamped.png')
result = extract(m3, reloaded)

print('Photo size:', photo.size, '| stamped size:', stamped.size)
print('Read back:', {k: v for k, v in result.items() if k != 'bits'})

difference = np.abs(np.asarray(stamped, dtype=np.float32) - np.asarray(photo, dtype=np.float32))
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for axis, image, title in zip(axes, [photo, stamped, np.clip(difference * 10, 0, 255).astype(np.uint8)],
                              ['Original', 'Stamped', 'Difference x10']):
    axis.imshow(image); axis.set_title(title); axis.axis('off')
plt.tight_layout()
plt.show()

## Check on all 500 test photos

Each photo is stamped with a random verdict and confidence, edited, and read back. For each edit the table shows how many bits were read correctly and how many photos gave a valid stamp with the right verdict. We also check how many photos without a stamp are wrongly read as valid.

At full size the stamp held up better than in the 128 pixel tests of notebook 3. After JPEG at quality 70, 99.58% of bits were correct and the verdict could be read for 90% of photos, compared with 92.08% of bits and 8.8% of photos before. The reason is that `extract` shrinks the photo before reading it, and shrinking smooths out most of the damage JPEG causes. No unstamped photo was read as valid.

The example photo above also shows a limit. In smooth areas such as the sky, faint lines from the stamp can be seen.

In [ ]:
def save_as(image, fmt, quality=None):
    buffer = io.BytesIO()
    image.save(buffer, format=fmt, **({'quality': quality} if quality else {}))
    buffer.seek(0)
    return Image.open(buffer).convert('RGB')

EDITS = {
    'PNG': lambda im: save_as(im, 'PNG'),
    'JPEG 90': lambda im: save_as(im, 'JPEG', 90),
    'JPEG 70': lambda im: save_as(im, 'JPEG', 70),
    'JPEG 50': lambda im: save_as(im, 'JPEG', 50),
    'Resized to 224': lambda im: save_as(im.resize((224, 224), Image.BILINEAR), 'PNG'),
}
rng = np.random.default_rng(0)
bits_ok = {name: [] for name in EDITS}
verdict_ok = {name: [] for name in EDITS}
psnr, unstamped_valid = [], 0

for path in test_files:
    photo = Image.open(path).convert('RGB')
    verdict = ['Real', 'AI', 'Uncertain'][rng.integers(3)]
    p_ai, tag = float(rng.random()), int(rng.integers(2**14))
    expected = encode_payload(verdict, p_ai, image_tag=tag)
    stamped = embed(m3, photo, verdict, p_ai, image_tag=tag)

    error = np.mean((np.asarray(save_as(stamped, 'PNG'), dtype=np.float32) - np.asarray(photo, dtype=np.float32)) ** 2)
    psnr.append(10 * np.log10(255 ** 2 / max(error, 1e-12)))
    unstamped_valid += extract(m3, save_as(photo, 'PNG'))['valid']

    for name, edit in EDITS.items():
        bits = read_bits(m3, edit(stamped))
        bits_ok[name].append(np.mean([a == b for a, b in zip(bits, expected)]))
        decoded = decode_payload(bits)
        verdict_ok[name].append(decoded['valid'] and decoded['verdict'] == verdict)

print(f'{"Edit":<16} {"Bits correct":>13} {"Valid, right verdict":>22}')
for name in EDITS:
    print(f'{name:<16} {np.mean(bits_ok[name]):>13.2%} {np.mean(verdict_ok[name]):>22.2%}')
print(f'PSNR at full size: {np.mean(psnr):.2f} dB')
print(f'Unstamped photos read as valid: {unstamped_valid} of {len(test_files)}')

## Notes for anyone using `m3_watermark.py`

Check `result['valid']` before using the verdict. A valid result means the message was read without errors. It does not prove the photo was never edited.

`embed` returns a PIL image. Saving it as PNG keeps the stamp exactly. Saving it as JPEG also works but costs some bits, as the table shows.

`read_bits` returns the raw 32 bits, which is useful for measuring bit accuracy after an edit.

Load the model once with `load_m3` and reuse it for all photos. Loading it for every photo is much slower.